<a href="https://colab.research.google.com/github/Progammingfrenzy/diabetes-ml-classification/blob/main/Diabetes_FT_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd
import numpy as np
import torch
import sklearn

print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("Scikit-learn", sklearn.__version__)

Pandas: 2.2.3
PyTorch: 2.11.0+cpu
Scikit-learn 1.6.1


In [7]:
import os

#Show files currently uploaded to Colab
print(os.listdir("/content"))

['.config', 'sample_data']


In [8]:
# Load dataset directly from GitHub
url = "https://raw.githubusercontent.com/razibdash/Diabetes-Health-Indicators/main/diabetes_012_health_indicators_BRFSS2015.csv"

df = pd.read_csv(url)

print("Dataset shape:", df.shape)


Dataset shape: (253680, 22)


In [9]:
# Check for Missing Values
print("Missing values:", df.isnull().sum().sum())

# Check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Display data types
print(df.dtypes)

Missing values: 0
Duplicate rows: 23899
Diabetes_012            float64
HighBP                  float64
HighChol                float64
CholCheck               float64
BMI                     float64
Smoker                  float64
Stroke                  float64
HeartDiseaseorAttack    float64
PhysActivity            float64
Fruits                  float64
Veggies                 float64
HvyAlcoholConsump       float64
AnyHealthcare           float64
NoDocbcCost             float64
GenHlth                 float64
MentHlth                float64
PhysHlth                float64
DiffWalk                float64
Sex                     float64
Age                     float64
Education               float64
Income                  float64
dtype: object


In [10]:

# Binary features (0 or 1)
binary_cols = [
    "HighBP", "HighChol", "CholCheck",
    "Smoker", "Stroke", "HeartDiseaseorAttack",
    "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare",
    "NoDocbcCost", "DiffWalk", "Sex"
]

# Ordinal features (ordered categories)
ordinal_cols = [
    "GenHlth", "Age", "Education", "Income"
]

# Numerical features
numerical_cols = [
    "BMI", "MentHlth", "PhysHlth"
]


# Define our input features and target
X = df.drop(columns=["Diabetes_012"])
y = df["Diabetes_012"].astype(int)

print("Features:", X.shape)
print("Target:", y.shape)

# Check that all 21 features are accounted for
all_cols = binary_cols + ordinal_cols + numerical_cols

assert len(all_cols) == len(set(all_cols))
assert set(all_cols) == set(X.columns)

print("Binary features:", len(binary_cols))
print("Ordinal features:", len(ordinal_cols))
print("Numerical features:", len(numerical_cols))
print("Total features:", len(all_cols))



Features: (253680, 21)
Target: (253680,)
Binary features: 14
Ordinal features: 4
Numerical features: 3
Total features: 21


In [11]:

from sklearn.model_selection import train_test_split

# Split into 80% training and 20% temporary data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Divide the temporary data into validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)


Training: (202944, 21)
Validation: (25368, 21)
Testing: (25368, 21)


In [12]:

# Show percentage of each diabetes class in training
print(y_train.value_counts(normalize=True) * 100)

Diabetes_012
0    84.240973
2    13.933400
1     1.825627
Name: proportion, dtype: float64


In [13]:

# Check validation class percentages
print(y_val.value_counts(normalize=True) * 100)

# Check testing class percentages
print(y_test.value_counts(normalize=True) * 100)


Diabetes_012
0    84.239987
2    13.934879
1     1.825134
Name: proportion, dtype: float64
Diabetes_012
0    84.243929
2    13.930937
1     1.825134
Name: proportion, dtype: float64


In [14]:

# Identify matching feature combinations
train_keys = pd.util.hash_pandas_object(X_train, index=False)
test_keys = pd.util.hash_pandas_object(X_test, index=False)

# Count matching test records
overlap = test_keys.isin(set(train_keys)).sum()

print("Overlapping test records:", overlap)


Overlapping test records: 3480


In [15]:

# Check whether our previous work is still loaded
print("Dataset loaded:", "df" in globals())
print("Training data ready:", "X_train" in globals())


Dataset loaded: True
Training data ready: True


In [17]:

from sklearn.preprocessing import StandardScaler

# Create a scaler
scaler = StandardScaler()

# Make copies of our datasets
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

# Scale numerical features
X_train_scaled[numerical_cols] = scaler.fit_transform(
    X_train[numerical_cols]
)

X_val_scaled[numerical_cols] = scaler.transform(
    X_val[numerical_cols]
)

X_test_scaled[numerical_cols] = scaler.transform(
    X_test[numerical_cols]
)

# View the scaled training features
display(X_train_scaled[numerical_cols].head())


,BMI,MentHlth,PhysHlth
185494,-0.512552,-0.429290,-0.485961
217030,1.301115,-0.429290,-0.485961
641,0.998837,-0.429290,-0.485961
217863,-0.210274,-0.429290,0.088138
53677,-0.059135,-0.023967,-0.485961


In [19]:
# Check the mean and standard deviation
print(X_train_scaled[numerical_cols].mean().round(2))
print(X_train_scaled[numerical_cols].std().round(2))


BMI        -0.0
MentHlth    0.0
PhysHlth    0.0
dtype: float64
BMI         1.0
MentHlth    1.0
PhysHlth    1.0
dtype: float64


In [20]:
# Combine binary and ordinal feature names
categorical_cols = binary_cols + ordinal_cols

# Check the feature counts
print("Categorical features:", len(categorical_cols))
print("Numerical features:", len(numerical_cols))


Categorical features: 18
Numerical features: 3


In [21]:

# Examine categorical values in the training dataset
category_summary = X_train[categorical_cols].agg(
    ["min", "max", "nunique"]
).T

display(category_summary)


,min,max,nunique
HighBP,0.0,1.0,2.0
HighChol,0.0,1.0,2.0
CholCheck,0.0,1.0,2.0
Smoker,0.0,1.0,2.0
Stroke,0.0,1.0,2.0
HeartDiseaseorAttack,0.0,1.0,2.0
PhysActivity,0.0,1.0,2.0
Fruits,0.0,1.0,2.0
Veggies,0.0,1.0,2.0
HvyAlcoholConsump,0.0,1.0,2.0
